In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
import joblib

# Create project folders
os.makedirs("/content/project/model", exist_ok=True)

# 1) Generate Realistic Customer Churn Dataset
np.random.seed(42)
n_samples = 1200

data = {
    'Age': np.random.randint(18, 70, size=n_samples),
    'Tenure_Months': np.random.randint(1, 72, size=n_samples),
    'Monthly_Bill': np.round(np.random.uniform(20.0, 150.0, size=n_samples), 2),
    'Total_Usage_GB': np.random.randint(10, 500, size=n_samples),
    'Support_Calls': np.random.randint(0, 10, size=n_samples),
    'Payment_Delay_Days': np.random.randint(0, 30, size=n_samples),
    'Multiple_Lines': np.random.choice([0, 1], size=n_samples),
    'Device_Protection': np.random.choice([0, 1], size=n_samples),
    'Tech_Support_Plan': np.random.choice([0, 1], size=n_samples),
    'Streaming_TV': np.random.choice([0, 1], size=n_samples),
    'Paperless_Billing': np.random.choice([0, 1], size=n_samples),
    'Has_Promo_Code': np.random.choice([0, 1], size=n_samples),
    'Contract_Length_Years': np.random.choice([0, 1, 2], size=n_samples),
    'Family_Plan_Members': np.random.randint(1, 6, size=n_samples)
}

# Design target with realistic feature dependencies
churn_score = (
    0.15 * data['Support_Calls']
    + 0.005 * data['Monthly_Bill']
    + 0.05 * data['Payment_Delay_Days']
    - 0.04 * data['Tenure_Months']
    + 0.1 * (30 - data['Age']) / 10
)
probs = 1 / (1 + np.exp(-churn_score))
data['Churn'] = (probs > 0.45).astype(int)

df_churn = pd.DataFrame(data)
print(f"Dataset successfully prepared! Shape: {df_churn.shape} (14 features, 1200 rows)")

# 2) Train-Test Split (80/20) & Save Evaluation Partition
X = df_churn.drop(columns=["Churn"])
y = df_churn["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_test_with_label = X_test.copy()
X_test_with_label["Churn"] = y_test
X_test_with_label.to_csv("/content/project/test_data.csv", index=False)
print("Saved evaluation partition to /content/project/test_data.csv")

# 3) Initialize, Fit and Export the 5 Required Model Pipelines
classifiers = {
    "Logistic_Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision_Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive_Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42)
}

results = []
for name, clf in classifiers.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', clf)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name.replace("_", " "),
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC Score": roc_auc_score(y_test, y_proba),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "MCC Score": matthews_corrcoef(y_test, y_pred)
    })
    joblib.dump(pipeline, f"/content/project/model/{name}.joblib")

metrics_df = pd.DataFrame(results)
display(metrics_df)

Dataset successfully prepared! Shape: (1200, 15) (14 features, 1200 rows)
Saved evaluation partition to /content/project/test_data.csv


,Model,Accuracy,AUC Score,Precision,Recall,F1 Score,MCC Score
0,Logistic Regression,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,Decision Tree,0.883333,0.910134,0.903030,0.925466,0.914110,0.732913
2,KNN,0.829167,0.902272,0.852941,0.900621,0.876133,0.603933
3,Naive Bayes,0.929167,0.977121,0.928571,0.968944,0.948328,0.837822
4,SVM,0.937500,0.992767,0.939759,0.968944,0.954128,0.857147


In [14]:
# ==========================================
# 2) Create app.py Streamlit Code
# ==========================================
app_code = """import streamlit as st
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report

st.set_page_config(page_title="Customer Churn Predictor", layout="wide")

st.title("Subscription Service - Customer Churn Predictor")
st.write("Upload validation test CSV containing subscriber features and the 'Churn' target label to evaluate pre-trained models dynamically.")

st.sidebar.header("Configuration")
model_options = {
    "Logistic Regression": "Logistic_Regression.joblib",
    "Decision Tree": "Decision_Tree.joblib",
    "KNN": "KNN.joblib",
    "Naive Bayes": "Naive_Bayes.joblib",
    "SVM": "SVM.joblib"
}
selected_model_name = st.sidebar.selectbox("Select Classifier Model", list(model_options.keys()))
uploaded_file = st.sidebar.file_uploader("Upload Churn Test CSV File", type=["csv"])

model_path = os.path.join("/content/project/model", model_options[selected_model_name])
if not os.path.exists(model_path):
    model_path = os.path.join("model", model_options[selected_model_name])

try:
    pipeline = joblib.load(model_path)
    st.sidebar.success(f"Loaded {selected_model_name} successfully!")
except Exception as e:
    st.sidebar.error(f"Could not load model pipeline: {e}")
    pipeline = None

if uploaded_file is not None and pipeline is not None:
    df = pd.read_csv(uploaded_file)
    st.subheader("Uploaded Subscriber Dataset Preview")
    st.dataframe(df.head(10))

    target_col = "Churn"
    if target_col not in df.columns:
        st.error(f"The dataset must contain the target column named '{target_col}'.")
    else:
        X_eval = df.drop(columns=[target_col])
        y_eval = df[target_col]

        y_pred = pipeline.predict(X_eval)
        y_proba = pipeline.predict_proba(X_eval)[:, 1] if hasattr(pipeline, "predict_proba") else y_pred

        acc = accuracy_score(y_eval, y_pred)
        auc = roc_auc_score(y_eval, y_proba)
        prec = precision_score(y_eval, y_pred, zero_division=0)
        rec = recall_score(y_eval, y_pred, zero_division=0)
        f1 = f1_score(y_eval, y_pred, zero_division=0)
        mcc = matthews_corrcoef(y_eval, y_pred)

        st.subheader(f"Evaluation Metrics - {selected_model_name}")
        col1, col2, col3, col4, col5, col6 = st.columns(6)
        col1.metric("Accuracy", f"{acc:.4f}")
        col2.metric("AUC Score", f"{auc:.4f}")
        col3.metric("Precision", f"{prec:.4f}")
        col4.metric("Recall", f"{rec:.4f}")
        col5.metric("F1 Score", f"{f1:.4f}")
        col6.metric("MCC Score", f"{mcc:.4f}")

        fig_col1, fig_col2 = st.columns(2)
        with fig_col1:
            st.write("**Confusion Matrix**")
            cm = confusion_matrix(y_eval, y_pred)
            fig, ax = plt.subplots(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", cbar=False,
                        xticklabels=["Active (0)", "Churned (1)"],
                        yticklabels=["Active (0)", "Churned (1)"])
            plt.ylabel("True Status")
            plt.xlabel("Predicted Status")
            st.pyplot(fig)

        with fig_col2:
            st.write("**Classification Report**")
            report_dict = classification_report(y_eval, y_pred, output_dict=True)
            report_df = pd.DataFrame(report_dict).transpose()
            st.dataframe(report_df.style.format(precision=4))
else:
    st.info("Please upload 'test_data.csv' from the sidebar to begin evaluation.")
"""

with open("/content/project/app.py", "w") as f:
    f.write(app_code)
print("Saved clean app.py successfully.")

Saved clean app.py successfully.


In [15]:
from google.colab import files
import os

file_path = '/content/project/app.py'
if os.path.exists(file_path):
    files.download(file_path)
    print("Direct download triggered for app.py!")
else:
    print("Error: /content/project/app.py not found. Please run the cell above to generate it first!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Direct download triggered for app.py!


In [19]:
# ==========================================
# 3) Create README.md & requirements.txt
# ==========================================
requirements_content = """streamlit>=1.20.0
scikit-learn>=1.0
numpy
pandas
matplotlib
seaborn
joblib
"""
with open("/content/project/requirements.txt", "w") as f:
    f.write(requirements_content)

readme_text = f"""# Subscription Service Customer Churn Predictor

## a. Problem Statement
Retaining active subscribers is a major operational challenge for modern digital service platforms. Identifying at-risk customers early allows companies to deploy targeted retention strategies. The goal of this project is to build a binary classification system using 5 machine learning models to predict whether a subscriber will churn (`1`) or remain active (`0`) based on customer demographic data and usage patterns.

## b. Dataset Description
- **Target Column**: `Churn` (Binary: 0 for Active, 1 for Churned)
- **Dataset Size**: 1200 instances (Exceeds required minimum of 500 instances)
- **Feature Size**: 14 input features (Exceeds required minimum of 12 features)

### Input Features:
1. **Age**: Numerical age of the subscriber (18 to 70 years).
2. **Tenure_Months**: Number of months the customer has been with the service.
3. **Monthly_Bill**: Monthly service cost (USD).
4. **Total_Usage_GB**: Internet bandwidth data usage in gigabytes.
5. **Support_Calls**: Frequency of customer care interactions.
6. **Payment_Delay_Days**: Number of days payment has been overdue.
7. **Multiple_Lines**: Binary flag representing subscription to multiple telephone lines.
8. **Device_Protection**: Binary flag indicating active equipment coverage plan.
9. **Tech_Support_Plan**: Binary flag representing prioritized tech support tier.
10. **Streaming_TV**: Binary flag representing subscription to video service streaming.
11. **Paperless_Billing**: Binary flag indicating electronic delivery of invoices.
12. **Has_Promo_Code**: Binary flag indicating registration using promotional codes.
13. **Contract_Length_Years**: Categorical contract tier (0: Month-to-month, 1: One-year, 2: Two-year).
14. **Family_Plan_Members**: Number of household members linked to the contract.

## c. GitHub Repository Link
- **GitHub Repository**: [https://github.com/<your-username>/customer-churn-classifier](https://github.com/<your-username>/customer-churn-classifier)
- **Live Streamlit App**: [https://customer-churn-classifier.streamlit.app](https://customer-churn-classifier.streamlit.app)

## d. Models Used & Performance Evaluation Table

| ML Model Name | Accuracy | AUC Score | Precision | Recall | F1 Score | MCC Score |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 1.0000 | 1.0000 | 1.0000 | 1.0000 | 1.0000 | 1.0000 |
| **Decision Tree** | 0.8833 | 0.9101 | 0.9030 | 0.9255 | 0.9141 | 0.7329 |
| **K-Nearest Neighbors (KNN)** | 0.8292 | 0.9023 | 0.8529 | 0.9006 | 0.8761 | 0.6039 |
| **Naive Bayes** | 0.9292 | 0.9771 | 0.9286 | 0.9689 | 0.9483 | 0.8378 |
| **Support Vector Machine (SVM)** | 0.9375 | 0.9928 | 0.9398 | 0.9689 | 0.9541 | 0.8571 |

### Overall Winner for your dataset?
**Support Vector Machine (SVM)** or **Decision Tree** are excellent choices. SVM achieves a highly balanced evaluation profile with an accuracy of **93.75%** and an outstanding AUC Score of **0.9928** on standard non-linear inputs.

### Observations on Model Performance
- **Logistic Regression**: Linear separation achieves perfect metrics on standard synthetic boundary thresholds.
- **Decision Tree**: Captures multi-level splits well with an accuracy of 88.33%.
- **KNN**: Strong distance-based baseline reaching an AUC score of 90.23%.
- **Naive Bayes**: Performs exceptionally with a 92.92% Accuracy and stable classification indicators.
- **Support Vector Machine (SVM)**: Robust non-linear model achieving 93.75% Accuracy and an outstanding AUC Score of 99.28%.
"""
with open("/content/project/README.md", "w") as f:
    f.write(readme_text)
print("Saved clean, single-pass README.md and requirements.txt successfully!")

Saved clean, single-pass README.md and requirements.txt successfully!


In [18]:
import os
import shutil
from google.colab import files

# Ensure the source project directory exists
if os.path.exists('/content/project'):
    # Generate zip archive dynamically
    shutil.make_archive('/content/project_repository', 'zip', '/content/project')
    print("Successfully packed /content/project_repository.zip!")

    # Trigger browser download
    try:
        files.download('/content/project_repository.zip')
        print("Download interface initiated successfully.")
    except Exception as e:
        print(f"Download interface failed: {e}. You can manually download 'project_repository.zip' from the left sidebar.")
else:
    print("Error: The directory '/content/project' does not exist. Please run the previous model training and app.py cells first!")

Successfully packed /content/project_repository.zip!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download interface initiated successfully.
